# MedGemma Service for MedFlow (Pro+ Optimized)

This notebook runs MedGemma model on Google Colab and exposes it via ngrok for MedFlow to use.

**Optimized for Colab Pro+:**
- 24-hour runtime support
- Background execution enabled
- Keep-alive mechanisms
- Resource monitoring

## ⚙️ Recommended Settings:
- **Runtime Type:** Python 3
- **GPU:** T4 GPU ✅ (Perfect for 4B model)
- **High-RAM:** OFF ❌ (Not needed, saves compute units)

## Instructions:
1. Get your Hugging Face token from: https://huggingface.co/settings/tokens
2. Get your ngrok auth token from: https://dashboard.ngrok.com/get-started/your-authtoken
3. Replace the tokens below
4. Runtime → Change runtime type → GPU (T4)
5. Run all cells
6. Copy the ngrok URL and set it as `MEDGEMMA_REMOTE_URL` in MedFlow

In [ ]:
# Install dependencies
!pip install -q transformers torch fastapi uvicorn pyngrok Pillow accelerate psutil gputil

In [ ]:
# Configuration - REPLACE THESE WITH YOUR TOKENS
HF_TOKEN = "hf_YOUR_HUGGINGFACE_TOKEN_HERE"  # Get from https://huggingface.co/settings/tokens
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # Get from https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# Setup keep-alive and session management
from IPython.display import display, Javascript, HTML
import time
from datetime import datetime
import threading

# Track session start time and requests
start_time = datetime.now()
request_count = 0

# Enable background execution - keeps session alive even when tab closed
display(Javascript('''
  function KeepAlive() {
    console.log("🔄 Keep-Alive System Activated");
    
    // Ping kernel every 5 minutes
    setInterval(() => {
      if (typeof google !== 'undefined' && google.colab && google.colab.kernel) {
        console.log("[" + new Date().toLocaleTimeString() + "] ✓ Session alive");
        // Trigger a keep-alive by accessing the kernel
        google.colab.kernel.proxyPort(8000, {'cache': false});
      }
    }, 5 * 60 * 1000); // 5 minutes
    
    // Prevent tab from going idle
    setInterval(() => {
      document.title = "🟢 MedGemma Service [Active] - " + new Date().toLocaleTimeString();
    }, 30 * 1000); // Update every 30 seconds
  }
  
  KeepAlive();
'''))

print("="*60)
print("🔄 SESSION KEEP-ALIVE ACTIVATED")
print("="*60)
print(f"✓ Background execution enabled")
print(f"✓ Session will stay alive even if you close browser tab")
print(f"✓ Started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✓ With Pro+: Up to 24 hours continuous runtime")
print("="*60 + "\n")

In [ ]:
# Resource monitoring setup
import psutil

try:
    import GPUtil
except:
    !pip install -q gputil
    import GPUtil

def check_resources():
    """Display current resource usage"""
    # RAM Usage
    ram = psutil.virtual_memory()
    print(f"💾 RAM: {ram.percent:.1f}% used ({ram.used/1024**3:.1f}GB / {ram.total/1024**3:.1f}GB)")
    
    # GPU Usage
    try:
        gpus = GPUtil.getGPUs()
        if gpus:
            gpu = gpus[0]
            print(f"🎮 GPU: {gpu.name}")
            print(f"   Memory: {gpu.memoryUsed}MB / {gpu.memoryTotal}MB ({gpu.memoryUtil*100:.1f}%)")
            print(f"   Utilization: {gpu.load*100:.1f}%")
    except Exception as e:
        print(f"⚠️  GPU info not available: {e}")

def monitor_loop():
    """Background monitoring every hour"""
    while True:
        time.sleep(3600)  # Every hour
        uptime = datetime.now() - start_time
        hours = uptime.seconds // 3600
        minutes = (uptime.seconds % 3600) // 60
        
        print(f"\n{'='*60}")
        print(f"📊 Resource Check: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"⏰ Uptime: {hours}h {minutes}m")
        print(f"{'='*60}")
        check_resources()
        print(f"{'='*60}\n")

# Start monitoring in background
monitor_thread = threading.Thread(target=monitor_loop, daemon=True)
monitor_thread.start()

print("✓ Resource monitoring started (checks every hour)")
print("✓ Initial resource status:")
check_resources()
print()

In [ ]:
# Imports and setup
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
from pyngrok import ngrok
from huggingface_hub import login
import uvicorn
import nest_asyncio
import torch
import base64
import io
from typing import Optional

nest_asyncio.apply()

# Login to Hugging Face
print("🔐 Logging in to Hugging Face...")
login(token=HF_TOKEN)
print("✓ Logged in to Hugging Face")

# Set ngrok auth token
print("🔐 Configuring ngrok...")
ngrok.set_auth_token(NGROK_TOKEN)
print("✓ ngrok configured")

In [ ]:
# Load MedGemma model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n{'='*60}")
print(f"📥 Loading MedGemma model on {device.upper()}...")
print(f"{'='*60}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️  WARNING: No GPU detected! Performance will be very slow.")

MODEL_NAME = "google/medgemma-1.5-4b-it"
print(f"Model: {MODEL_NAME}")

processor = AutoProcessor.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto"
)

print(f"\n{'='*60}")
print(f"✅ Model loaded successfully on {device.upper()}!")
print(f"{'='*60}\n")

In [ ]:
# Create FastAPI app with request tracking
app = FastAPI(title="MedGemma Inference Service for MedFlow")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Request models
class TextRequest(BaseModel):
    text: str
    max_new_tokens: Optional[int] = 512

class ImageRequest(BaseModel):
    image_base64: str
    prompt: Optional[str] = "Describe this medical image in detail."
    max_new_tokens: Optional[int] = 512

class MultimodalRequest(BaseModel):
    text: str
    image_base64: str
    max_new_tokens: Optional[int] = 512

# Helper functions
def decode_image(base64_string: str) -> Image.Image:
    """Decode base64 image string to PIL Image"""
    image_data = base64.b64decode(base64_string)
    return Image.open(io.BytesIO(image_data)).convert('RGB')

def generate_response(messages, max_new_tokens=512):
    """Generate response from MedGemma model"""
    global request_count
    request_count += 1
    
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )
        generation = generation[0][input_len:]

    return processor.decode(generation, skip_special_tokens=True)

# API Endpoints
@app.get("/")
async def root():
    uptime = datetime.now() - start_time
    return {
        "status": "MedGemma running",
        "model": MODEL_NAME,
        "device": device,
        "uptime_seconds": uptime.seconds,
        "requests_served": request_count
    }

@app.post("/predict_text")
async def predict_text(request: TextRequest):
    """Process text-only medical analysis"""
    try:
        print(f"[TEXT] Request #{request_count + 1}: {request.text[:100]}...")

        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": "You are a helpful medical AI assistant."}]
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": request.text}]
            }
        ]

        response = generate_response(messages, request.max_new_tokens)
        print(f"[TEXT] Response generated: {response[:100]}...")

        return {
            "input": request.text,
            "response": response,
            "mode": "text-only"
        }

    except Exception as e:
        print(f"[TEXT] ERROR: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict_image")
async def predict_image(request: ImageRequest):
    """Process medical image analysis"""
    try:
        print(f"[IMAGE] Request #{request_count + 1}: {request.prompt[:100]}")
        image = decode_image(request.image_base64)
        print(f"[IMAGE] Image size: {image.size}")

        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": "You are an expert medical imaging assistant."}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": request.prompt}
                ]
            }
        ]

        response = generate_response(messages, request.max_new_tokens)
        print(f"[IMAGE] Response generated: {response[:100]}...")

        return {
            "prompt": request.prompt,
            "response": response,
            "mode": "image-only"
        }

    except Exception as e:
        print(f"[IMAGE] ERROR: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict_multimodal")
async def predict_multimodal(request: MultimodalRequest):
    """Process text + image multimodal analysis"""
    try:
        print(f"[MULTIMODAL] Request #{request_count + 1}: {request.text[:100]}...")
        image = decode_image(request.image_base64)
        print(f"[MULTIMODAL] Image size: {image.size}")

        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": "You are an expert medical imaging assistant."}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": request.text}
                ]
            }
        ]

        response = generate_response(messages, request.max_new_tokens)
        print(f"[MULTIMODAL] Response generated: {response[:100]}...")

        return {
            "input": request.text,
            "response": response,
            "mode": "multimodal"
        }

    except Exception as e:
        print(f"[MULTIMODAL] ERROR: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

print("✓ FastAPI app created with request tracking")

In [ ]:
# Periodic status logging
def log_periodic_status():
    """Log status every 30 minutes"""
    while True:
        time.sleep(1800)  # 30 minutes
        uptime = datetime.now() - start_time
        hours = uptime.seconds // 3600
        minutes = (uptime.seconds % 3600) // 60
        
        print(f"\n{'='*60}")
        print(f"[{datetime.now().strftime('%H:%M:%S')}] 🟢 STATUS CHECK")
        print(f"{'='*60}")
        print(f"✓ Service Running")
        print(f"✓ Uptime: {hours}h {minutes}m")
        print(f"✓ Requests Served: {request_count}")
        print(f"✓ Model: {MODEL_NAME}")
        print(f"✓ Device: {device.upper()}")
        print(f"{'='*60}\n")

# Start status logging in background
status_thread = threading.Thread(target=log_periodic_status, daemon=True)
status_thread.start()
print("✓ Status logging started (every 30 minutes)")

In [ ]:
# Start the server with ngrok tunnel
print("\n" + "="*60)
print("🚀 Starting MedGemma service...")
print("="*60 + "\n")

# Create ngrok tunnel
public_url = ngrok.connect(8000)

# Display prominent notice
print("\n" + "#"*60)
print("#" + " "*58 + "#")
print("#" + "  ⚠️  IMPORTANT: COPY THIS URL TO MEDFLOW  ⚠️".center(58) + "#")
print("#" + " "*58 + "#")
print("#"*60)
print()
print(f"  PUBLIC URL: {public_url}")
print()
print("#"*60)
print()
print("📋 COPY AND USE:")
print(f"   export MEDGEMMA_REMOTE_URL='{public_url}'")
print()
print("📝 OR ADD TO .env FILE:")
print(f"   MEDGEMMA_REMOTE_URL={public_url}")
print()
print("#"*60 + "\n")

# Display runtime information
display(HTML(f"""
<div style="padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
            border-radius: 15px; color: white; margin: 20px 0; box-shadow: 0 4px 15px rgba(0,0,0,0.2);">
    <h2 style="margin: 0 0 15px 0;">🎉 MedGemma Service is Running!</h2>
    <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; margin: 10px 0;">
        <h3 style="margin: 0 0 10px 0;">📡 Your Public URL:</h3>
        <code style="background: rgba(0,0,0,0.3); padding: 10px; display: block; 
                     border-radius: 5px; font-size: 16px;">{public_url}</code>
    </div>
    <p style="margin: 15px 0 0 0;">✅ Service Status: <strong>ACTIVE</strong></p>
    <p style="margin: 5px 0;">⏰ Started: <strong>{start_time.strftime('%Y-%m-%d %H:%M:%S')}</strong></p>
    <p style="margin: 5px 0;">💾 Model: <strong>{MODEL_NAME}</strong></p>
    <p style="margin: 5px 0;">🎮 Device: <strong>{device.upper()}</strong></p>
    <p style="margin: 15px 0 0 0; font-size: 14px; opacity: 0.9;">
        ℹ️ You can close this browser tab - the service will keep running with Pro+!
    </p>
</div>
"""))

# Start uvicorn server in a thread
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

print("✓ Server started!")
print("\n" + "="*60)
print("🎯 NEXT STEPS FOR MEDFLOW:")
print("="*60)
print("1. Copy the URL above")
print("2. Update MedFlow .env file: MEDGEMMA_REMOTE_URL=<url>")
print("3. Restart MedFlow: ./restart_services.sh")
print("4. Test connection: python test_medgemma_connection.py <url>")
print("="*60)
print()
print("💡 TIP: Leave this tab open (can minimize). Service runs in background!")
print("⏰ With Pro+: Up to 24 hours continuous runtime")
print("🔄 Status updates appear here every 30 minutes")
print()
print("🎉 Ready to serve MedFlow requests!\n")

---

## 📊 Service Running

The service is now active and ready to receive requests from MedFlow!

### Status Indicators:
- You'll see status updates every 30 minutes in the output above
- Request logs appear when MedFlow makes requests
- Resource checks appear every hour

### Keeping Service Running:
- ✅ Keep this browser tab open (can minimize)
- ✅ Keep-alive is automatic with Pro+
- ✅ Service continues even if screen goes to sleep
- ✅ Up to 24 hours continuous runtime

### Monitoring:
Watch the output above for:
- Request logs: `[TEXT] Request #X`
- Status updates: Every 30 minutes
- Resource checks: Every hour

### If Service Stops:
1. Restart: Runtime → Run all
2. Wait 3 minutes for model to load
3. Copy new ngrok URL
4. Update MedFlow .env file
5. Restart MedFlow: `./restart_services.sh`

---